In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import os
from tqdm import tqdm
from datasets import load_from_disk

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct", use_fast=False)
model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
            model_name,  
            device_map="cuda:0",  
            torch_dtype=torch.bfloat16,  
            trust_remote_code=True,  
) 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Code

In [3]:
num_seeds = 10

In [5]:
__DIR__ = "../knowledge_critical/KG_networks/data/preprocessed/"

In [6]:
dataset_folders = [os.path.join(__DIR__, f) for f in os.listdir(__DIR__)]
print(dataset_folders)
dataset = load_from_disk(dataset_folders[0])

['../knowledge_critical/KG_networks/data/preprocessed/code_generated', '../knowledge_critical/KG_networks/data/preprocessed/natural_instructions', '../knowledge_critical/KG_networks/data/preprocessed/apps', '../knowledge_critical/KG_networks/data/preprocessed/astronomy_generated', '../knowledge_critical/KG_networks/data/preprocessed/general_generated']


In [7]:
dataset = dataset.map(lambda x: {"tokenized":tokenizer.apply_chat_template([{"role":"user", "content":x["user"]},{"role":"assistant","content":x["assistant"]}], padding="max_length", truncation=True, max_length=4096, return_tensors="pt")[0]})

In [8]:
dataset.set_format("torch", columns=["tokenized"])

In [9]:
columns_to_remove = ["user", "assistant", "role"]
dataset = dataset.remove_columns(columns_to_remove)

In [10]:
dataset

Dataset({
    features: ['tokenized'],
    num_rows: 197
})

In [11]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [12]:
model.eval()

for param in model.parameters():
    param.requires_grad = True

In [13]:
assistant_token = 32001

In [14]:
def get_importance_score(dataloader):
    
    head_importance = torch.zeros(num_hidden_layers, num_heads).to(model.device)

    for batch in tqdm(dataloader):
        batch = batch["tokenized"]
        inputs = batch[:,1:].to(model.device)
        labels = batch[:,:-1].to(model.device)
        indices = (labels == assistant_token).nonzero(as_tuple=True)
        mask = (torch.cumsum(torch.ones_like(labels),dim=-1)-1)<=indices[1].view(-1,1)
        labels[mask] = -100
        outputs = model(input_ids=inputs, labels=labels, output_attentions=True)

        for attn in outputs.attentions:
            attn.retain_grad()

        outputs.loss.backward()

        head_importance_ = torch.zeros_like(head_importance)

        for layer, attn in enumerate(outputs.attentions):
            for head in range(num_heads):
                head_importance_[layer,head] += torch.bmm(attn[:,head].transpose(-2,-1),attn.grad[:,head]).abs().mean()
        print(head_importance_.mean())
        head_importance_ /= head_importance_.sum()
        head_importance += head_importance_


        for param in model.parameters():
            if param.grad is not None:
                param.grad.zero_()
                
    return head_importance/len(dataloader)

In [14]:
# Set the model to evaluation mode
model.eval()

# Enable gradient tracking for attention weights
for param in model.parameters():
    param.requires_grad = True

outputs = model(input_ids=torch.tensor([[1,2,3,4]]).cuda(), labels=torch.tensor([[2,3,4,5]]).cuda(), output_attentions=True)


You are not running the flash-attention implementation, expect numerical differences.


In [15]:
for attn in outputs.attentions:
    attn.retain_grad()

In [16]:
outputs.loss.backward()

In [28]:
head_importance = torch.zeros(num_hidden_layers, num_heads).to(model.device)

# Sum gradients over tokens to get head importance
for layer, attn in enumerate(outputs.attentions):
    # Average the gradients across tokens for each head
    for head in range(num_heads):
        head_importance[layer,head] += torch.bmm(attn[:,head].transpose(-2,-1),attn.grad[:,head]).abs().mean()


In [20]:
for batch in dataloader:
    print(batch.shape)
    break

AttributeError: 'dict' object has no attribute 'shape'

In [23]:
batch["tokenized"]

tensor([[32000, 32000, 32000,  ...,    13, 32007, 32000],
        [32000, 32000, 32000,  ..., 29897, 32007, 32000]])

In [15]:
head_importance = get_importance_score(dataloader)
head_importance

  0%|          | 0/197 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 23.68 GiB of which 1.89 GiB is free. Including non-PyTorch memory, this process has 21.78 GiB memory in use. Of the allocated memory 21.16 GiB is allocated by PyTorch, and 322.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [18]:
head_importance /= head_importance.sum()

In [46]:
import plotly.graph_objects as go

heatmap = go.Heatmap(z=head_importance.detach().cpu().numpy())
fig = go.Figure(data=heatmap)
fig.show()